In [1]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
from pathlib import Path
from time import perf_counter
import inspect
import re
import urllib.request
import zipfile

import numpy as np
import pandas as pd
import torch
from PIL import Image, ImageDraw
from IPython.display import display
from tqdm.auto import tqdm

from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from ultralytics import SAM

In [3]:
PROJECT = Path("/content/drive/MyDrive/Vision/vision_unit_04")
MANIFEST = PROJECT / "outputs/evaluation_manifest.csv"
DATA = Path("/content/datasets/coco128-seg")

manifest = pd.read_csv(MANIFEST)
cal_names = manifest.loc[manifest.split == "calibration", "image_name"].tolist()
test_names = manifest.loc[manifest.split == "test", "image_name"].tolist()

print(f"Calibration: {len(cal_names)} images | Test: {len(test_names)} images")

Calibration: 20 images | Test: 20 images


In [5]:
CLASSES = {0: "person", 2: "car", 16: "dog", 41: "cup", 56: "chair"}
NAME_TO_ID = {name: cid for cid, name in CLASSES.items()}
PROMPTS = {
    "combined": ("chair. cup. dog. car. person.",),
    "separate": ("chair.", "cup.", "dog.", "car.", "person."),
}
BOX_THRESHOLD, TEXT_THRESHOLD = 0.35, 0.30
MODEL_ID = "IDEA-Research/grounding-dino-tiny"

In [6]:
processor = AutoProcessor.from_pretrained(MODEL_ID)
detector = AutoModelForZeroShotObjectDetection.from_pretrained(MODEL_ID)
detector = detector.to("cuda").float().eval()
sam = SAM("sam2.1_s.pt")

postprocess = processor.post_process_grounded_object_detection
box_argument = (
    "threshold" if "threshold" in inspect.signature(postprocess).parameters
    else "box_threshold"
)

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


Loading weights:   0%|          | 0/978 [00:00<?, ?it/s]

**Read Image and GT**